In [1]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import re
from urllib.parse import urlparse
import time
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../dataset/balanced_links.csv')
def normalize_url(url):
    url = str(url).strip()
    url = url.replace('https://', '').replace('http://', '')
    return url

df['url'] = df['url'].apply(normalize_url)
print("URLs normalized!")
print(df['url'].head(5).to_string())
print(df.shape)
df.head()

URLs normalized!
0                                       asamcgill.com/
1    www.bogdanobradovic.com/t3-assets/js/2014/abt/...
2    noticias.uol.com.br/tabloide/ultimas-noticias/...
3        manta.com/c/mt4crnc/sue-kelly-dermatology-llc
4    ofmenandangels.blogspot.com/2009/02/vince-azzo...
(446176, 3)


,url,type,label
0,asamcgill.com/,benign,0
1,www.bogdanobradovic.com/t3-assets/js/2014/abt/...,phishing,1
2,noticias.uol.com.br/tabloide/ultimas-noticias/...,benign,0
3,manta.com/c/mt4crnc/sue-kelly-dermatology-llc,benign,0
4,ofmenandangels.blogspot.com/2009/02/vince-azzo...,benign,0


In [2]:
# sample of URLs to fetch HTML from
sample_size = 10000  

benign_sample = df[df['label'] == 0].sample(n=sample_size//2, random_state=42)
malicious_sample = df[df['label'] == 1].sample(n=sample_size//2, random_state=42)

df_sample = pd.concat([benign_sample, malicious_sample]).reset_index(drop=True)

print("Sample shape:", df_sample.shape)
print(df_sample['label'].value_counts())

Sample shape: (10000, 3)
label
0    5000
1    5000
Name: count, dtype: int64


In [3]:
def extract_html_features(url):
    features = {}
    
    url = str(url)
    if not url.startswith('http'):
        url = 'http://' + url
    
    try:
        response = requests.get(url, timeout=5, verify=False,
                                headers={'User-Agent': 'Mozilla/5.0'})
        soup = BeautifulSoup(response.content, 'html.parser')
        
        # Count tags
        features['num_iframes'] = len(soup.find_all('iframe'))
        features['num_scripts'] = len(soup.find_all('script'))
        features['num_images'] = len(soup.find_all('img'))
        features['num_links'] = len(soup.find_all('a'))
        features['num_forms'] = len(soup.find_all('form'))
        features['num_inputs'] = len(soup.find_all('input'))
        
        # Password field
        password_inputs = soup.find_all('input', {'type': 'password'})
        features['has_password_field'] = 1 if len(password_inputs) > 0 else 0
        
        # External resources
        scripts = soup.find_all('script', src=True)
        features['num_external_scripts'] = sum(
            1 for s in scripts if s['src'].startswith('http')
        )
        
        imgs = soup.find_all('img', src=True)
        features['num_external_images'] = sum(
            1 for i in imgs if i['src'].startswith('http')
        )
        
        # Hidden elements
        hidden = soup.find_all(style=re.compile(r'display\s*:\s*none'))
        features['num_hidden_elements'] = len(hidden)
        
        # Page size
        features['page_size'] = len(response.content)
        features['fetch_status'] = 1  # Successfully fetched
        
    except:
        # If page can't be fetched, fill with zeros
        features = {
            'num_iframes': 0, 'num_scripts': 0, 'num_images': 0,
            'num_links': 0, 'num_forms': 0, 'num_inputs': 0,
            'has_password_field': 0, 'num_external_scripts': 0,
            'num_external_images': 0, 'num_hidden_elements': 0,
            'page_size': 0, 'fetch_status': 0  # Failed to fetch
        }
    
    return features

In [4]:
print("Fetching HTML features... this will take a few minutes")
print("Total URLs to fetch:", len(df_sample))

html_features_list = []

for i, row in df_sample.iterrows():
    if i % 50 == 0:
        print(f"Progress: {len(html_features_list)}/{len(df_sample)}")
    features = extract_html_features(row['url'])
    features['url'] = row['url']
    features['label'] = row['label']
    html_features_list.append(features)
    time.sleep(0.1)  # delay to avoid getting blocked

df_html = pd.DataFrame(html_features_list)
print("\nDone! Shape:", df_html.shape)
print("Successfully fetched:", df_html['fetch_status'].sum(), "out of", len(df_html))

Fetching HTML features... this will take a few minutes
Total URLs to fetch: 10000
Progress: 0/10000


Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


Progress: 50/10000


Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.
Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.
Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


Progress: 100/10000


...

Progress: 9900/10000


Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.
Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.
Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


Progress: 9950/10000


Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.
Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.



Done! Shape: (10000, 14)
Successfully fetched: 6407 out of 10000


In [5]:
df_html.to_csv('../dataset/html_features.csv', index=False)
print("Saved successfully!")
print("\nFetch status breakdown:")
print(df_html['fetch_status'].value_counts())
print("\nLabel breakdown:")
print(df_html['label'].value_counts())

Saved successfully!

Fetch status breakdown:
fetch_status
1    6407
0    3593
Name: count, dtype: int64

Label breakdown:
label
0    5000
1    5000
Name: count, dtype: int64


In [6]:
# Load URL features
df_url = pd.read_csv('../dataset/url_features.csv')
df_url_sample = df_url.loc[df_sample.index].reset_index(drop=True)

# Reset html features index
df_html_reset = df_html.reset_index(drop=True)

# Merge both feature sets
df_combined = pd.concat([df_url_sample.drop('label', axis=1), df_html_reset], axis=1)

print("Combined shape:", df_combined.shape)
print("Columns:", df_combined.columns.tolist())
df_combined.head()

Combined shape: (10000, 43)
Columns: ['url_length', 'domain_length', 'path_length', 'num_dots', 'num_hyphens', 'num_slashes', 'num_at', 'num_digits', 'num_params', 'num_percent', 'num_subdomains', 'has_ip', 'has_https', 'has_at_symbol', 'has_double_slash', 'has_login', 'has_verify', 'has_secure', 'has_account', 'has_update', 'has_banking', 'has_confirm', 'has_password', 'has_signin', 'has_ebayisapi', 'has_webscr', 'has_cmd', 'has_dispatch', 'has_paypal', 'num_iframes', 'num_scripts', 'num_images', 'num_links', 'num_forms', 'num_inputs', 'has_password_field', 'num_external_scripts', 'num_external_images', 'num_hidden_elements', 'page_size', 'fetch_status', 'url', 'label']


,url_length,domain_length,path_length,num_dots,num_hyphens,num_slashes,num_at,num_digits,num_params,num_percent,...,num_forms,num_inputs,has_password_field,num_external_scripts,num_external_images,num_hidden_elements,page_size,fetch_status,url,label
0,14,13,1,1,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,thecanadianencyclopedia.com/index.cfm?PgNm=TCE...,0
1,56,23,33,3,1,5,0,5,0,0,...,0,0,0,11,1,0,77418,1,soundcloud.com/a-trak/robyn-indestructible-a-t...,0
2,146,19,127,4,13,7,0,8,0,0,...,0,0,0,0,0,0,1530,1,linkedin.com/pub/peter-wege/12/336/926,0
3,45,9,36,1,3,3,0,1,0,0,...,0,0,0,0,0,0,9805,1,chacha.com/question/how-much-are-charles-barkl...,0
4,56,27,29,3,1,3,0,6,0,0,...,1,2,0,1,1,0,195061,1,toronto-theatre.com/theaters/bluma-appel-theat...,0


In [7]:
df_combined.to_csv('../dataset/combined_features.csv', index=False)
print("Saved successfully!")

Saved successfully!
